# Delta Lake Internals

Delta Lake internals is the foundation that everything else (DLT, streaming, Unity Catalog) sits on.

We'll cover it in 4 layers, from most fundamental to most advanced:

| Layer | Topic | Why It Matters |
|---|---|---|
| 1 | Transaction Log (`_delta_log`) | The "brain" of Delta Lake: how ACID actually works |
| 2 | Time Travel | The visible feature of the transaction log |
| 3 | OPTIMIZE + ZORDER + VACUUM | File management for performance and cost |
| 4 | Checkpoints & Recovery | What happens when things break |

We'll do hands-on practice for Layers 2 and 3 (they're the most testable in Free Edition). Layers 1 and 4 are theory-heavy but crucial for interviews.

## Layer 1: The Transaction Log (`_delta_log`)

### The Simple Explanation

When you save a Delta table, Databricks doesn't just save your Parquet files. It also saves a log of every change.

```text
/my_table/
├── _delta_log/                                  ← THE BRAIN (metadata)
│   ├── 00000000000000000000.json                ← version 0 (CREATE)
│   ├── 00000000000000000001.json                ← version 1 (INSERT)
│   ├── 00000000000000000002.json                ← version 2 (UPDATE)
│   ├── 00000000000000000003.json                ← version 3 (DELETE)
│   └── 00000000000000000010.checkpoint.parquet  ← snapshot
│
├── part-00000-abc.parquet                       ← actual data files
├── part-00001-def.parquet
└── part-00002-ghi.parquet
```

### What's Inside Each JSON?

Each `00000000000000000001.json` file contains actions:

| Action | Meaning |
|---|---|
| `add` | "This Parquet file is now part of the table" |
| `remove` | "This Parquet file is no longer part of the table" |
| `metaData` | Schema, partitioning, table properties |
| `commitInfo` | Who did what, when, with what operation |

Every commit is atomic: either the whole JSON is written, or it isn't. That's how ACID works on cloud storage.

`This shows every operation ever performed on the table — with version numbers, timestamps, operations (WRITE, MERGE, OPTIMIZE), and metrics (rows added/removed).`

In [0]:
%sql
-- Create table (version 0)
CREATE OR REPLACE TABLE workspace.default.time_travel_demo (
  id INT,
  value STRING
) USING DELTA;

INSERT INTO workspace.default.time_travel_demo VALUES (1, 'original');

-- Check the version
DESCRIBE HISTORY workspace.default.time_travel_demo;

In [0]:
%sql
-- Update (new version)
UPDATE workspace.default.time_travel_demo SET value = 'updated' WHERE id = 1;

-- Insert more rows (new version)
INSERT INTO workspace.default.time_travel_demo VALUES (2, 'second row');

-- Check the version
DESCRIBE HISTORY workspace.default.time_travel_demo;

```DESCRIBE DETAIL workspace.default.your_table_name;``` **to check location column**

##Layer 2 : Time Travel
`Because every change is in the log, you can query any past version of your table.`

In [0]:
%sql
SELECT * FROM workspace.default.time_travel_demo;

In [0]:
%sql
-- See what it looked like at version 1 (before any changes)
SELECT * FROM workspace.default.time_travel_demo VERSION AS OF 1;

###What Kills Time Travel?
`VACUUM. It physically deletes old Parquet files. After VACUUM, time travel to versions older than the retention period (default 7 days) stops working — the underlying data is gone. `

**This is a critical interview point:** "Time travel is only as good as VACUUM allows it to be."

## Layer 3: OPTIMIZE, ZORDER, VACUUM

### The Problem: Small Files

Streaming and frequent writes create many small Parquet files. This slows down queries (file listing overhead) and increases storage cost.

### OPTIMIZE: Combine Small Files

```sql
-- Bin-pack: merge small files into larger ones (default target: ~1GB)
OPTIMIZE workspace.default.time_travel_demo;
```

- **Idempotent:** running it twice on the same data does nothing extra.
- **Doesn't change data:** it just rearranges files.
- **Readers aren't interrupted** (snapshot isolation).

### ZORDER: Co-locate Related Data

```sql
-- ZORDER: physically group rows with similar values together
OPTIMIZE workspace.default.titanic
ZORDER BY (Sex, Pclass);
```

**Why:** If you often query `WHERE Sex = 'female'`, ZORDER puts all female rows in the same files, so data skipping reads fewer files.

**Rule:** ZORDER works best for low-to-medium cardinality columns (few distinct values). Don't ZORDER on `user_id` with millions of distinct values.

### VACUUM: Delete Old Files

```sql
-- Delete files no longer referenced in the log, older than 7 days
VACUUM workspace.default.time_travel_demo;

-- See what would be deleted (dry run)
VACUUM workspace.default.time_travel_demo DRY RUN;
```

> **Warning:** VACUUM kills time travel beyond the retention window (default 7 days).


## Layer 4: Checkpoints & Recovery

### Why Checkpoints Exist

After thousands of versions, reading every JSON file from version 0 is slow. Delta Lake periodically writes a **checkpoint**: a Parquet file summarizing the entire table state at that version (by default every 10 commits).

```text
_delta_log/
├── 000...000.json                  ← version 0
├── 000...001.json                  ← version 1
├── ...
├── 000...010.checkpoint.parquet    ← snapshot at version 10
├── 000...011.json                  ← version 11
├── ...
└── _last_checkpoint                ← points readers to the latest checkpoint
```

**To reconstruct version 15:**

- If a checkpoint at version 10 exists → read checkpoint + JSONs 11–15 (fast)
- If no checkpoint → read JSONs 0–15 (slow)


####Delta Lake automatically writes checkpoints every 10 commits (versions).


## Layer 4 Practical: Checkpoints & Recovery

### Step 1: Make 15 commits

```python
spark.sql("CREATE OR REPLACE TABLE workspace.default.checkpoint_demo (id INT, val STRING)")
for i in range(1, 16):
    spark.sql(f"INSERT INTO workspace.default.checkpoint_demo VALUES ({i}, 'row_{i}')")
```

### Step 2: See the versions

```sql
DESCRIBE HISTORY workspace.default.checkpoint_demo;
```

You should see versions 0 to 15.

### Step 3: Look at the checkpoint files

```python
loc = spark.sql("DESCRIBE DETAIL workspace.default.checkpoint_demo").first()["location"]
display(dbutils.fs.ls(f"{loc}/_delta_log"))
```

Look for `00000000000000000010.checkpoint.parquet` and `_last_checkpoint`.

> **Note:** Free Edition blocks this on managed tables (`DBFS_DISABLED`, location is null). Skip it, or write the Delta table to a Volume path to see the log files.

### Step 4: Break the table

```sql
DELETE FROM workspace.default.checkpoint_demo WHERE id > 5;
UPDATE workspace.default.checkpoint_demo SET val = 'CORRUPTED';
SELECT * FROM workspace.default.checkpoint_demo;
```

Only 5 rows left, all `CORRUPTED`.

### Step 5: Check the good version

```sql
SELECT * FROM workspace.default.checkpoint_demo VERSION AS OF 15;
```

You should get 15 clean rows.

### Step 6: Recover

```sql
RESTORE TABLE workspace.default.checkpoint_demo TO VERSION AS OF 15;
SELECT COUNT(*) FROM workspace.default.checkpoint_demo;
```

Count is back to 15.

### Step 7: Confirm RESTORE is a new commit

```sql
DESCRIBE HISTORY workspace.default.checkpoint_demo;
```

The top row is `RESTORE`, and the `DELETE` and `UPDATE` are still in history.
~~~

# Change Data Feed (CDF)

Change Data Feed (CDF) records every row-level change (insert, update, delete) made to a Delta table, so downstream jobs can read only what changed instead of rescanning the whole table.

Each row comes back with `_change_type` (insert, update_preimage, update_postimage, delete), `_commit_version`, and `_commit_timestamp`. It is used for incremental ETL from Silver to Gold, SCD Type 2, and syncing changes to other systems. It only captures changes made after you enable it.

## The Problem

Say you have an orders table with 10 million rows. Every night, a few hundred orders are added, some are updated, and a few are deleted. Your Gold table needs those changes.

Without CDF, you either re-read all 10 million rows or build your own `last_updated` tracking, which can miss deletes.

## The Idea

CDF is like a CCTV camera on your table. Once it is enabled, Delta records every row that gets inserted, updated, or deleted, along with the version and timestamp when it happened.

Later, you can ask: "Show me everything that changed since version 2."

## Example

Version 1: You insert 3 orders.

| id | status |
| -- | ------ |
| 1  | placed |
| 2  | placed |
| 3  | placed |

Version 2: Order 2 is shipped, and order 3 is deleted.

Asking CDF for changes from version 2 returns:

| id | status  | _change_type     | _commit_version |
| -- | ------- | ---------------- | --------------- |
| 2  | placed  | update_preimage  | 2               |
| 2  | shipped | update_postimage | 2               |
| 3  | placed  | delete           | 2               |

Order 1 isn't there because it didn't change.

## The Four Change Types

* `insert`: A new row was added.
* `update_preimage`: What the row looked like before the update.
* `update_postimage`: What the row looks like after the update.
* `delete`: The row that was removed.

You get both the before and after values of an update, which is useful for SCD Type 2 and audit trails.

## Enable and Read CDF

### 1. Turn it on

Only changes made after enabling CDF are recorded.

```sql
ALTER TABLE workspace.default.orders
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = true);
```

### 2. Read changes from a version onward

```sql
SELECT *
FROM table_changes('workspace.default.orders', 2);
```

### 3. Read changes between versions

```sql
SELECT *
FROM table_changes('workspace.default.orders', 2, 5);
```

### 4. Read changes since a timestamp

```sql
SELECT *
FROM table_changes(
    'workspace.default.orders',
    '2026-10-12 00:00:00'
);
```

### 5. Read CDF using PySpark

```python
df = (
    spark.read
    .option("readChangeFeed", "true")
    .option("startingVersion", 2)
    .table("workspace.default.orders")
)
```

CDF can also be read as a stream using Structured Streaming.

```
CDC (Change Data Capture): This is a concept or a process. 
It means capturing changes (Insert, Update, Delete) made to a source database (like PostgreSQL or SQL Server) 
so you can replicate them downstream.

CDF (Change Data Feed): This is a specific Databricks Delta Lake feature. 
When you turn it on (delta.enableChangeDataFeed = true), 
Databricks writes row-level change data into the table's transaction log so you can query it.
```

## Where CDF Is Used

### 1. Silver to Gold: Process Only Changed Rows

Suppose your Gold table contains total sales per customer, built from a Silver orders table with 10 million rows.

Today, only 300 orders changed.

Without CDF, you might recalculate everything from all 10 million rows every night, which is slow and expensive.

With CDF, you read only the changed rows and update the relevant customer totals.

### 2. SCD Type 2

SCD Type 2 maintains historical records when data changes.

The preimage provides the old values, and the postimage provides the new values. These can help you close the previous record and insert a new version.

### 3. Syncing Other Systems

You can send only inserted, updated, and deleted rows to another warehouse or system instead of transferring the entire table.

### 4. Auditing Changes

CDF helps you inspect which rows changed and when. You can combine it with Delta table history using `DESCRIBE HISTORY` for additional information about table operations.

---

# Liquid Clustering

Liquid Clustering is a modern Delta Lake technique for organizing data to improve query performance. It offers a flexible alternative to traditional partitioning and Z-Ordering.

You choose clustering keys, and Delta organizes related rows together in data files as data arrives and optimization runs.

## The Problem

Your orders table is stored as hundreds of Parquet files.

When you run:

```sql
SELECT *
FROM workspace.default.orders
WHERE order_date = '2026-10-12';
```

Databricks tries to read only the files that contain rows for that date.

If the date's rows are scattered across many files, Databricks may need to scan more data than necessary, making the query slower.

The goal is to keep similar values organized so Databricks can skip irrelevant files.

Historically, there were two common approaches.

## Old Way 1: Partitioning

```sql
CREATE TABLE workspace.default.orders (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DOUBLE
)
USING DELTA
PARTITIONED BY (order_date);
```

Partitioning creates separate storage directories for different partition values.

For example:

```text
orders/
    order_date=2026-10-10/
    order_date=2026-10-11/
    order_date=2026-10-12/
```

### Problems with Partitioning

* It works well for suitable columns, such as dates.
* High-cardinality columns such as `customer_id` can create too many partitions and small files.
* Changing the partitioning strategy can require substantial data rewriting or migration.

## Old Way 2: Z-Ordering

```sql
OPTIMIZE workspace.default.orders
ZORDER BY (customer_id);
```

Z-Ordering reorganizes data to improve data skipping for queries that filter on the selected columns.

### Problems with Z-Ordering

* It requires maintenance using `OPTIMIZE`.
* New data may not benefit until optimization runs.
* You must choose and maintain the Z-Ordering columns.
* Reorganizing data consumes compute resources.

## New Way: Liquid Clustering

Liquid Clustering provides a flexible way to organize Delta table data for faster queries.

You define the columns that are commonly used in filters, and Delta incrementally organizes the data when clustering operations run.

### Create a Table with Liquid Clustering

```sql
CREATE TABLE workspace.default.orders (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DOUBLE
)
USING DELTA
CLUSTER BY (order_date, customer_id);
```

### Optimize the Table

```sql
OPTIMIZE workspace.default.orders;
```

`OPTIMIZE` reorganizes eligible data to improve clustering. With incremental clustering, it can avoid repeatedly reorganizing data that is already well-clustered.

Databricks may also support automatic optimization, depending on the table configuration and environment.

## Why Liquid Clustering Is Better

### 1. Change Clustering Keys

You can change the clustering columns without requiring an immediate full-table rewrite.

```sql
ALTER TABLE workspace.default.orders
CLUSTER BY (customer_id);
```

Existing data may be reorganized gradually through subsequent optimization.

### 2. Supports High-Cardinality Columns

Liquid Clustering can work well with columns such as `customer_id` and `user_id`, which may have many distinct values.

### 3. Avoids Partition Directories for Every Value

Unlike partitioning, Liquid Clustering does not create a separate directory for every clustering-key value. This avoids the directory explosion associated with high-cardinality partitioning.

### 4. Incremental Maintenance

Clustering can focus on data that needs reorganization instead of repeatedly rewriting the entire table.

### 5. Automatic Clustering Keys

Where supported, `CLUSTER BY AUTO` allows Databricks to select clustering keys based on observed query patterns.

```sql
CREATE TABLE workspace.default.orders (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DOUBLE
)
USING DELTA
CLUSTER BY AUTO;
```

## Summary

| Feature      | Change Data Feed (CDF)                       | Liquid Clustering                       |
| ------------ | -------------------------------------------- | --------------------------------------- |
| Main purpose | Capture row-level changes                    | Improve query performance               |
| Solves       | Reprocessing entire tables                   | Scanning unnecessary data files         |
| Common use   | Incremental ETL, SCD Type 2, synchronization | Faster filtering and data skipping      |
| How it works | Records inserts, updates, and deletes        | Organizes data based on clustering keys |
| Main benefit | Process only changed rows                    | Read less data for suitable queries     |


# MERGE INTO (Upsert)

`MERGE INTO` combines multiple operations in one statement:

* UPDATE existing rows
* INSERT new rows
* DELETE rows missing from the source (optional)

## Example: Syncing Phone Contacts

Imagine syncing your phone contacts with a company directory.

* Contact exists in both → update with the latest information.
* Contact only in the directory → insert it.
* Contact only on your phone → delete it if it is no longer in the company directory.

### Phone (Target) Before Sync

| id | name  | phone      |
| -- | ----- | ---------- |
| 1  | Ravi  | 9000000001 |
| 2  | Sita  | 9000000002 |
| 4  | Kiran | 9000000004 |

### Company Directory (Source) Today

| id | name  | phone      |
| -- | ----- | ---------- |
| 1  | Ravi  | 9111111111 |
| 2  | Sita  | 9000000002 |
| 3  | Arjun | 9000000003 |

Ravi exists in both tables but has a new number, so update him.

Arjun exists only in the directory, so insert him.

Kiran exists only on the phone because he left the company, so delete him.

### SQL Syntax

```sql
MERGE INTO workspace.default.phone_contacts AS t
USING workspace.default.company_directory AS s
ON t.id = s.id

WHEN MATCHED THEN
    -- Exists in both: update with the latest information
    UPDATE SET
        t.name = s.name,
        t.phone = s.phone

WHEN NOT MATCHED THEN
    -- Exists only in the source: insert
    INSERT (id, name, phone)
    VALUES (s.id, s.name, s.phone)

WHEN NOT MATCHED BY SOURCE THEN
    -- Exists only in the target: delete
    DELETE;
```

**Important:** `WHEN NOT MATCHED BY SOURCE THEN DELETE` can delete every target row that has no matching source row. Use it only when the source represents the complete dataset that should be synchronized, or add an appropriate condition.

---

# Table Constraints

## What They Are

Constraints are rules enforced when data is written to a table.

Two common types are:

| Type       | Meaning                                 |
| ---------- | --------------------------------------- |
| `NOT NULL` | A column cannot contain NULL values.    |
| `CHECK`    | A condition must be true for every row. |

### NOT NULL

```sql
CREATE TABLE people (
    id INT NOT NULL,
    name STRING NOT NULL
) USING DELTA;
```

Both `id` and `name` must have non-NULL values.

### CHECK

```sql
ALTER TABLE people
ADD CONSTRAINT age_check CHECK (age > 0);
```

This requires `age` to be greater than zero for rows that are written.

The `age` column must already exist in the table.

## Key Points

* Constraints are enforced during writes. Invalid rows cause the write to fail.
* Delta Lake supports these constraints.
* CHECK constraints can be added or dropped later, subject to validation of existing data and platform rules.
* `NOT NULL` is declared in the table schema. Changing nullability later may require schema changes or other supported operations.

## Constraints vs. DLT Expectations

| Feature     | Table Constraints         | DLT Expectations                                             |
| ----------- | ------------------------- | ------------------------------------------------------------ |
| Where       | Supported Delta tables    | Declarative pipeline environments                            |
| Purpose     | Enforce table-level rules | Monitor and enforce data quality                             |
| Enforcement | Reject invalid writes     | Warn, drop invalid rows, or fail, depending on configuration |
| Visibility  | Constraint errors         | Data quality metrics and pipeline results                    |

**Simple rule:** Use constraints to protect table integrity. Use pipeline expectations to define and monitor data quality rules in supported pipeline environments.

---

# Generated Columns

## What They Are

A generated column has a value computed automatically from other columns using a defined expression.

### Example

```sql
CREATE TABLE people (
    id INT,
    birthDate TIMESTAMP,
    yearOfBirth INT GENERATED ALWAYS AS (YEAR(birthDate))
) USING DELTA;
```

Here, `yearOfBirth` is calculated from `birthDate` rather than supplied independently.

For example, if `birthDate` is `1998-05-20`, then `yearOfBirth` is `1998`.

The supported expressions and generated-column features depend on the Databricks and Delta Lake environment.

---

# Identity Columns

## What They Are

An identity column automatically generates numeric values for inserted rows, similar to `AUTO_INCREMENT` in MySQL or `IDENTITY` in SQL Server.

### Example

```sql
CREATE TABLE events (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    name STRING
) USING DELTA;
```

The database generates the `id` value when a row is inserted.

## When to Use

Use identity columns when you need automatically generated numeric IDs and their write-concurrency limitations are acceptable.

In Databricks Delta tables, identity columns can disable concurrent transactions that write to the same table. Consider this limitation before using them in high-concurrency workloads.

**Alternative:** Use UUIDs when globally unique identifiers are needed without relying on sequential numbers. `monotonically_increasing_id()` can generate distributed numeric IDs in Spark, but these are not guaranteed to be consecutive or stable across different executions.

---

# Idempotent Writes

## The Problem

Suppose your ETL job fails during a write. You restart it, and the second run writes the same data again.

If the first run already wrote some rows, the retry may create duplicates.

## The Solution

Delta Lake supports transaction identifiers for certain write operations to make retries idempotent.

| Option       | Meaning                                                |
| ------------ | ------------------------------------------------------ |
| `txnAppId`   | A string identifying the application or writing job    |
| `txnVersion` | A transaction version associated with that application |

## How It Works

Delta tracks transaction identifiers.

If the same application ID and transaction version are submitted again while the transaction record is retained, Delta can recognize the duplicate transaction and skip applying it again.

The application must manage these values consistently. Increment the transaction version for each new logical write and reuse the same version when retrying that write.

## PySpark Syntax

```python
df.write \
    .option("txnVersion", "5") \
    .option("txnAppId", "my-etl-job") \
    .mode("append") \
    .saveAsTable("events")
```

If this is a retry of transaction version `5` for the same application, Delta can avoid applying the same transaction twice.

---

# Schema Validation

## What It Is

Delta Lake validates the schema of incoming data when writing to an existing table.

This helps prevent incompatible data from being written accidentally.

## Common Rules

| Rule                   | What It Means                                                                                                  |
| ---------------------- | -------------------------------------------------------------------------------------------------------------- |
| Unexpected columns     | Extra columns generally cause an error unless schema evolution is enabled or another supported option applies. |
| Missing columns        | Missing target columns may be filled with NULL or defaults when supported and permitted by the schema.         |
| Data types             | Incompatible types can cause errors. Supported implicit casts may be allowed in some situations.               |
| Duplicate column names | Duplicate names are generally rejected, subject to the applicable schema and column-mapping rules.             |

### Example

Suppose the target table has this schema:

```text
id      INT
name    STRING
age     INT
```

But your DataFrame has:

```text
id      INT
name    STRING
```

The missing `age` column may be populated with NULL if the target column permits it and the write operation supports that behavior.

If the DataFrame contains an incompatible value for an integer column, the write may fail.

**Key point:** Schema validation protects the target table from incompatible writes. Schema evolution can allow certain changes, but it should be enabled deliberately.

---

# Selective Overwrite (replaceWhere)

## What It Does

`replaceWhere` overwrites only the rows matching a specified condition instead of replacing the entire table.

This is useful when refreshing a particular date range or business segment.

## Example

Suppose you need to refresh January 2026 data in an events table.

```python
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "replaceWhere",
        "date >= '2026-01-01' AND date <= '2026-01-31'"
    ) \
    .saveAsTable("events")
```

This replaces the rows matching the January date condition with the rows in the DataFrame, subject to the applicable `replaceWhere` validation settings.

Rows outside that date range remain unchanged.

**Important:** Make sure the incoming DataFrame contains the intended replacement data and that its rows satisfy the condition. Otherwise, the write may fail validation or replace the wrong subset if the condition is incorrect.

---

# Delta Lake Cloning

## What It Is

Cloning creates a new Delta table based on an existing table, optionally using a specific historical version.

Clones are useful for testing, development, backups, and reproducing a table state.

## The Two Types

| Type          | What It Copies                                          | Speed          | Storage                                     |
| ------------- | ------------------------------------------------------- | -------------- | ------------------------------------------- |
| Deep Clone    | Table metadata and independent copies of the data files | Usually slower | Requires additional storage for copied data |
| Shallow Clone | Table metadata and references to the source data files  | Usually faster | Requires less additional storage initially  |

## Syntax

### 1. Deep Clone

```sql
CREATE TABLE db.target
DEEP CLONE db.source;
```

Creates a separate copy of the source table's data files and metadata.

### 2. Shallow Clone

```sql
CREATE TABLE db.target
SHALLOW CLONE db.source;
```

Creates a clone that references the source table's existing data files.

### 3. Clone a Specific Version

```sql
CREATE TABLE db.target
SHALLOW CLONE db.source VERSION AS OF 5;
```

Creates a shallow clone based on version 5 of the source table.

**Important:** A shallow clone depends on the referenced source files remaining available. Source retention and cleanup operations can affect its long-term usability. A deep clone is more independent but requires additional storage and copy work.

---

# Partitioning Guidelines

## General Rules

| Rule                                       | Why                                                          |
| ------------------------------------------ | ------------------------------------------------------------ |
| Avoid very small partitions                | They can create many small files and increase overhead.      |
| Consider table size and query patterns     | Many tables do not need partitioning.                        |
| Prefer suitable, lower-cardinality columns | Columns such as date or region can be useful partition keys. |
| Avoid high-cardinality partition columns   | Columns such as `user_id` can create too many partitions.    |
| Avoid partitioning by precise timestamps   | Many distinct values can produce excessive partitions.       |

A partition size of around 1 GB or more is sometimes used as a rough guideline, not a strict requirement. The appropriate size depends on workload, data volume, and file layout.

## Analogy: Organizing a Filing Cabinet

Good: Organizing files by year.

```text
2024/
2025/
2026/
```

Bad: Creating a separate folder for every minute of the year, which would result in 525,600 folders.

The second approach creates unnecessary organizational overhead.

## Modern Recommendation

For new Delta tables, consider Liquid Clustering instead of traditional partitioning when it suits your workload.

Liquid Clustering provides flexible data organization without requiring a separate partition directory for every key value. It is particularly useful when query patterns change or the table uses high-cardinality filter columns.

Traditional partitioning can still be appropriate for certain large tables and workloads, so choose based on actual query patterns and data characteristics.


In [0]:
%sql
--cloning
DROP TABLE IF EXISTS workspace.default.orders_source;

CREATE TABLE workspace.default.orders_source (
  order_id INT,
  customer STRING,
  amount INT
)
USING DELTA;

INSERT INTO workspace.default.orders_source VALUES
  (1, 'Alice', 100),
  (2, 'Bob',   200),
  (3, 'Carol', 300);

SELECT * FROM workspace.default.orders_source;

In [0]:
%sql
--Add a Second Version (So We Can Clone at a Version Too)
INSERT INTO workspace.default.orders_source VALUES (4, 'David', 400);

-- Confirm two versions exist
DESCRIBE HISTORY workspace.default.orders_source;

In [0]:
%sql
--Deep clone copies data + metadata:
DROP TABLE IF EXISTS workspace.default.orders_deep;

CREATE TABLE workspace.default.orders_deep
DEEP CLONE workspace.default.orders_source;

SELECT * FROM workspace.default.orders_deep;

In [0]:
%sql
--Shallow clone copies only metadata and references the source's data files:
DROP TABLE IF EXISTS workspace.default.orders_shallow;

CREATE TABLE workspace.default.orders_shallow
SHALLOW CLONE workspace.default.orders_source;

SELECT * FROM workspace.default.orders_shallow;

**If source is deleted:** Deep still works, Shallow breaks